# 02 · Consolidación de datasets y catálogo de esquemas

Objetivos:

1. Homologar las tres fuentes en un **dataset maestro** (`data/processed/master_dataset.parquet`).
2. Construir la partición de BIRD por base de datos (semilla 42) y conservar las particiones originales de Spider.
3. Construir el **catálogo homogéneo de esquemas** (`data/processed/schemas.json`).
4. Validar particiones e integridad.

Reglas aplicadas (ver `src/data_loader.py`): no se traducen preguntas, no se altera el SQL, se conservan las evidencias vacías de BIRD,
los duplicados se marcan pero no se eliminan y los identificadores son reproducibles (`<SOURCE>-<split_original>-<índice>`).


In [2]:
# Configuración común: localizar la raíz del proyecto y habilitar `src/`
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

from src import config
config.ensure_directories()
print("Raíz del proyecto:", ROOT.name)

Raíz del proyecto: proyecto_jarm


## 1. Construcción del dataset maestro

In [5]:
from src.data_loader import build_master_dataset

master = build_master_dataset(save=True)
print("Columnas:", list(master.columns))
master.head(3)

2026-11-06 21:54:48,100 | INFO | src.data_loader | Spider train: 7000 ejemplos, 140 bases
2026-11-06 21:54:48,150 | INFO | src.data_loader | Spider dev: 1034 ejemplos, 20 bases
2026-11-06 21:54:48,194 | INFO | src.data_loader | BIRD train: 9428 ejemplos, 69 bases, 645 evidencias vacías
2026-11-06 21:54:48,212 | INFO | src.data_loader | BIRD dev: 1534 ejemplos, 11 bases, 148 evidencias vacías
2026-11-06 21:54:48,225 | INFO | src.data_loader | Partición BIRD cargada desde data/processed/bird_db_split.json
2026-11-06 21:54:48,820 | INFO | src.data_loader | Catastro: 80 pares (columnas detectadas: pregunta='Regla', sql='SQL (unidad_intervencion_18)')
2026-11-06 21:54:49,071 | INFO | src.data_loader | Duplicados exactos pregunta+SQL marcados: 30 registros en 15 grupos
2026-11-06 21:54:49,075 | INFO | src.data_loader | Dataset maestro: 19076 registros
2026-11-06 21:54:49,079 | INFO | src.data_loader | Distribución por fuente/split:
source    split        
BIRD      test             1432
    

,example_id,source,db_id,split,source_split,source_file,source_index,question,sql_gold,evidence,dialect,schema_id,schema_reference,difficulty,is_duplicate_question_sql,duplicate_group_id,source_row_number
0,SPIDER-train-00000,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,0,How many heads of the departments are older than 56 ?,SELECT count(*) FROM head WHERE age > 56,None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
1,SPIDER-train-00001,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,1,"List the name, born state and age of the heads of departments ordered by age.","SELECT name , born_state , age FROM head ORDER BY age",None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
2,SPIDER-train-00002,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,2,"List the creation year, name and budget of each department.","SELECT creation , name , budget_in_billions FROM department",None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN


In [6]:
master.groupby(["source", "split", "source_split"]).agg(n_examples=("example_id", "size"), n_dbs=("db_id", "nunique")).reset_index()

,source,split,source_split,n_examples,n_dbs
0,BIRD,test,train,1432,14
1,BIRD,train,train,7996,55
2,BIRD,validation,dev,1534,11
3,CATASTRO,test_transfer,catastro,80,1
4,SPIDER,train,train,7000,140
5,SPIDER,validation,dev,1034,20


### 1.1 Partición de BIRD por base de datos

La propuesta define train = 7.996 ejemplos / 55 bases, validation = 1.534 / 11 (dev público) y prueba interna = 1.432 / 14 bases reservadas del train original con semilla 42.
Como no existía ninguna partición persistida, se verificaron distintos procedimientos reproducibles; el único que reproduce exactamente esas cifras es:

```python
rng = np.random.default_rng(42)
test_dbs = [sorted(train_db_ids)[i] for i in rng.permutation(69)][:14]
```

La selección se persiste en `data/processed/bird_db_split.json` y se reutiliza en ejecuciones posteriores.

In [7]:
split_info = json.load(open(config.BIRD_SPLIT_PATH, encoding="utf-8"))
print("Semilla:", split_info["seed"], "| bases de prueba interna:", split_info["n_test_dbs"])
print(split_info["test_dbs"])
bird = master[master.source == "BIRD"]
check = bird.groupby("split").agg(n=("example_id", "size"), dbs=("db_id", "nunique"))
assert check.loc["test", "n"] == 1432 and check.loc["train", "n"] == 7996 and check.loc["validation", "n"] == 1534
check

Semilla: 42 | bases de prueba interna: 14
['beer_factory', 'books', 'craftbeer', 'cs_semester', 'genes', 'human_resources', 'mental_health_survey', 'movie_platform', 'music_platform_2', 'shooting', 'synthea', 'talkingdata', 'trains', 'world_development_indicators']


,n,dbs
split,,
test,1432,14
train,7996,55
validation,1534,11


### 1.2 Duplicados y trazabilidad

In [8]:
dups = master[master.is_duplicate_question_sql]
print(f"Registros duplicados exactos (pregunta + SQL dentro de la misma base): {len(dups)} en {dups.duplicate_group_id.nunique()} grupos (se conservan y se marcan)")
dups.sort_values("duplicate_group_id")[["example_id", "source", "db_id", "split", "duplicate_group_id", "question"]].head(6)

Registros duplicados exactos (pregunta + SQL dentro de la misma base): 30 en 15 grupos (se conservan y se marcan)


,example_id,source,db_id,split,duplicate_group_id,question
15854,BIRD-train-07820,BIRD,world,train,09e79b92ab,List any five countries which use English as an official language.
15864,BIRD-train-07830,BIRD,world,train,09e79b92ab,List any five countries which use English as an official language.
3413,SPIDER-train-03413,SPIDER,hr_1,train,124d9eb030,Display all the information about the department Marketing.
3483,SPIDER-train-03483,SPIDER,hr_1,train,124d9eb030,display all the information about the department Marketing.
10734,BIRD-train-02700,BIRD,regional_sales,train,20713597c0,How many sales teams are there in the Midwest?
10750,BIRD-train-02716,BIRD,regional_sales,train,20713597c0,How many sales teams are there in the Midwest?


In [9]:
print("Trazabilidad: cada registro conserva archivo e índice de origen y referencia a su esquema")
master[["example_id", "source_file", "source_index", "schema_id", "schema_reference"]].sample(5, random_state=config.RANDOM_SEED)

Trazabilidad: cada registro conserva archivo e índice de origen y referencia a su esquema


,example_id,source_file,source_index,schema_id,schema_reference
11887,BIRD-train-03853,raw/bird_data/train.json,3853,BIRD:public_review_platform,raw/bird_data/train_tables.json#public_review_platform
15978,BIRD-train-07944,raw/bird_data/train.json,7944,BIRD:music_platform_2,raw/bird_data/train_tables.json#music_platform_2
6383,SPIDER-train-06383,raw/spider_data/train_spider.json,6383,SPIDER:flight_company,raw/spider_data/tables.json#flight_company
7333,SPIDER-dev-00333,raw/spider_data/dev.json,333,SPIDER:cre_Doc_Template_Mgt,raw/spider_data/tables.json#cre_Doc_Template_Mgt
7800,SPIDER-dev-00800,raw/spider_data/dev.json,800,SPIDER:world_1,raw/spider_data/tables.json#world_1


In [ ]:
## Para modelo de linea base para genercion de texto se pasa en bruto pregunta y que arroje el SQL. 
master

,example_id,source,db_id,split,source_split,source_file,source_index,question,sql_gold,evidence,dialect,schema_id,schema_reference,difficulty,is_duplicate_question_sql,duplicate_group_id,source_row_number
0,SPIDER-train-00000,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,0,How many heads of the departments are older than 56 ?,SELECT count(*) FROM head WHERE age > 56,None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
1,SPIDER-train-00001,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,1,"List the name, born state and age of the heads of departments ordered by age.","SELECT name , born_state , age FROM head ORDER BY age",None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
2,SPIDER-train-00002,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,2,"List the creation year, name and budget of each department.","SELECT creation , name , budget_in_billions FROM department",None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
3,SPIDER-train-00003,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,3,What are the maximum and minimum budget of the departments?,"SELECT max(budget_in_billions) , min(budget_in_billions) FROM department",None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
4,SPIDER-train-00004,SPIDER,department_management,train,train,raw/spider_data/train_spider.json,4,What is the average number of employees of the departments whose rank is between 10 and 15?,SELECT avg(num_employees) FROM department WHERE ranking BETWEEN 10 AND 15,None,sqlite,SPIDER:department_management,raw/spider_data/tables.json#department_management,None,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19071,CATASTRO-catastro-00075,CATASTRO,catastro_unidad_intervencion_18,test_transfer,catastro,raw/catastro_data/catastral_SQL.xlsx,75,Direcciones asociadas en la tabla extdireccion de tipo Estructurada que no tienen diligenciada la clase de vía princ...,"SELECT\n ed.t_id AS direccion_id,\n ed.lc_predio_direccion AS predio_id,\n td.ilicode AS tipo_direccion,\n ...",None,postgres,CATASTRO:catastro_unidad_intervencion_18,raw/catastro_data/bd_catastro_test.sql#unidad_intervencion_18,None,False,NaN,76.0
19072,CATASTRO-catastro-00076,CATASTRO,catastro_unidad_intervencion_18,test_transfer,catastro,raw/catastro_data/catastral_SQL.xlsx,76,Predios cuyos datos adicionales de levantamiento catastral registran NIT como tipo de documento del reconocedor predial,"SELECT\n lp.t_id AS predio_id,\n lp.numero_predial,\n da.t_id AS datos_adicionales_id,\n dt.ilicode AS t...",None,postgres,CATASTRO:catastro_unidad_intervencion_18,raw/catastro_data/bd_catastro_test.sql#unidad_intervencion_18,None,False,NaN,77.0
19073,CATASTRO-catastro-00077,CATASTRO,catastro_unidad_intervencion_18,test_transfer,catastro,raw/catastro_data/catastral_SQL.xlsx,77,"Agrupaciones de interesados de tipo Grupo_Civil que integran personas jurídicas, de tipo Grupo_Empresarial que integ...","WITH composicion AS (\n SELECT\n ag.t_id AS agrupacion_id,\n ag.nombre,\n gt.ilicode AS tipo...",None,postgres,CATASTRO:catastro_unidad_intervencion_18,raw/catastro_data/bd_catastro_test.sql#unidad_intervencion_18,None,False,NaN,78.0
19074,CATASTRO-catastro-00078,CATASTRO,catastro_unidad_intervencion_18,test_transfer,catastro,raw/catastro_data/catastral_SQL.xlsx,78,"Fuentes administrativas clasificadas con tipo Sin_Documento que tienen diligenciado el ente emisor, el número de fue...","SELECT\n fa.t_id AS fuente_id,\n ft.ilicode AS tipo_fuente,\n fa.ente_emisor,\n fa.numero_fuente,\n f...",None,postgres,CATASTRO:catastro_unidad_intervencion_18,raw/catastro_data/bd_catastro_test.sql#unidad_intervencion_18,None,False,NaN,79.0


In [7]:
print("Evidencias de BIRD conservadas (vacías incluidas):")
master[master.source == "BIRD"].evidence.fillna("").str.strip().eq("").groupby(master.split).agg(["sum", "size"])

Evidencias de BIRD conservadas (vacías incluidas):


,sum,size
split,,
test,91,1432
train,554,7996
validation,148,1534


## 2. Catálogo homogéneo de esquemas

In [12]:
from src.schema_parser import build_schema_catalog, schema_summary

catalog = build_schema_catalog(save=True, required_schema_ids=set(master.schema_id))
summary = schema_summary(catalog)
summary.groupby("source").agg(n_schemas=("schema_id", "size"), tables_mean=("n_tables", "mean"), columns_mean=("n_columns", "mean"), fks_total=("n_foreign_keys", "sum"), issues=("n_issues", "sum")).round(2)

2026-11-06 21:56:38,818 | INFO | src.schema_parser | Spider: 166 esquemas
2026-11-06 21:56:38,902 | INFO | src.schema_parser | BIRD: 80 esquemas
2026-11-06 21:56:39,212 | INFO | src.schema_parser | Catastro: 232 tablas, 1821 columnas, 228 PK, 319 FK, 697 comentarios, 1 vistas (no candidatas)
2026-11-06 21:56:39,212 | INFO | src.schema_parser | Catálogo: 247 esquemas, 0 incidencias registradas
2026-11-06 21:56:39,385 | INFO | src.schema_parser | Guardado data/processed/schemas.json


,n_schemas,tables_mean,columns_mean,fks_total,issues
source,,,,,
BIRD,80,7.46,54.21,526,0
CATASTRO,1,232.00,1821.00,319,0
SPIDER,166,5.28,27.13,795,0


In [13]:
s = catalog["SPIDER:concert_singer"]
print("Campos del esquema:", list(s.keys()))
print("\nTablas:"); display(pd.DataFrame(s["tables"]))
print("Columnas (primeras 8):"); display(pd.DataFrame(s["columns"]).head(8))
print("PK:", s["primary_keys"]); print("FK:", s["foreign_keys"]); print("Relaciones:", s["relations"])
print("\nDDL reconstruido desde los metadatos (sin inventar restricciones):\n"); print(s["ddl"][:700])

Campos del esquema: ['schema_id', 'db_id', 'source', 'dialect', 'pg_schema', 'tables', 'columns', 'primary_keys', 'foreign_keys', 'relations', 'ddl', 'ddl_origin', 'n_tables', 'n_columns', 'issues']

Tablas:


,table_index,name_original,name_normalized,element_id,description,n_columns
0,0,stadium,stadium,stadium,None,7
1,1,singer,singer,singer,None,7
2,2,concert,concert,concert,None,5
3,3,singer_in_concert,singer in concert,singer_in_concert,None,2


Columnas (primeras 8):


,column_index,table_index,table_name,name_original,name_normalized,column_type,element_id,is_primary_key,description
0,1,0,stadium,Stadium_ID,stadium id,number,stadium.Stadium_ID,True,None
1,2,0,stadium,Location,location,text,stadium.Location,False,None
2,3,0,stadium,Name,name,text,stadium.Name,False,None
3,4,0,stadium,Capacity,capacity,number,stadium.Capacity,False,None
4,5,0,stadium,Highest,highest,number,stadium.Highest,False,None
5,6,0,stadium,Lowest,lowest,number,stadium.Lowest,False,None
6,7,0,stadium,Average,average,number,stadium.Average,False,None
7,8,1,singer,Singer_ID,singer id,number,singer.Singer_ID,True,None


PK: [{'table': 'stadium', 'columns': ['Stadium_ID']}, {'table': 'singer', 'columns': ['Singer_ID']}, {'table': 'concert', 'columns': ['concert_ID']}, {'table': 'singer_in_concert', 'columns': ['concert_ID']}]
FK: [{'from_table': 'concert', 'from_column': 'Stadium_ID', 'to_table': 'stadium', 'to_column': 'Stadium_ID'}, {'from_table': 'singer_in_concert', 'from_column': 'Singer_ID', 'to_table': 'singer', 'to_column': 'Singer_ID'}, {'from_table': 'singer_in_concert', 'from_column': 'concert_ID', 'to_table': 'concert', 'to_column': 'concert_ID'}]
Relaciones: [{'table_a': 'concert', 'table_b': 'stadium', 'via': ['concert.Stadium_ID -> stadium.Stadium_ID']}, {'table_a': 'singer_in_concert', 'table_b': 'singer', 'via': ['singer_in_concert.Singer_ID -> singer.Singer_ID']}, {'table_a': 'singer_in_concert', 'table_b': 'concert', 'via': ['singer_in_concert.concert_ID -> concert.concert_ID']}]

DDL reconstruido desde los metadatos (sin inventar restricciones):

CREATE TABLE "stadium" (
  "Stadium_

In [10]:
c = catalog[f"CATASTRO:{config.CATASTRO_DB_ID}"]
print(f"Catastro: {c['n_tables']} tablas, {c['n_columns']} columnas, origen DDL = {c['ddl_origin']}, esquema PostgreSQL = {c['pg_schema']}")
print("Relaciones FK de lc_predio:"); display(pd.DataFrame([fk for fk in c["foreign_keys"] if fk["from_table"] == "lc_predio"]))
print("Descripciones (comentarios del dump) disponibles:", sum(1 for col in c["columns"] if col["description"]), "columnas y", sum(1 for t in c["tables"] if t["description"]), "tablas")

Catastro: 232 tablas, 1821 columnas, origen DDL = pg_dump, esquema PostgreSQL = unidad_intervencion_18
Relaciones FK de lc_predio:


,from_table,from_column,to_table,to_column
0,lc_predio,categoria_suelo,lc_categoriasuelotipo,t_id
1,lc_predio,clase_suelo,lc_clasesuelotipo,t_id
2,lc_predio,condicion_predio,lc_condicionprediotipo,t_id
3,lc_predio,destinacion_economica,lc_destinacioneconomicatipo,t_id
4,lc_predio,tipo,col_unidadadministrativabasicatipo,t_id


Descripciones (comentarios del dump) disponibles: 612 columnas y 85 tablas


In [11]:
print("Esquemas requeridos por el dataset maestro y ausentes en el catálogo:", sorted(set(master.schema_id) - set(catalog)))
print("Esquemas con incidencias registradas:", summary[summary.n_issues > 0].schema_id.tolist())

Esquemas requeridos por el dataset maestro y ausentes en el catálogo: []
Esquemas con incidencias registradas: []


In [14]:
catalog

{'SPIDER:perpetrator': {'schema_id': 'SPIDER:perpetrator',
  'db_id': 'perpetrator',
  'source': 'SPIDER',
  'dialect': 'sqlite',
  'pg_schema': None,
  'tables': [{'table_index': 0,
    'name_original': 'perpetrator',
    'name_normalized': 'perpetrator',
    'element_id': 'perpetrator',
    'description': None,
    'n_columns': 8},
   {'table_index': 1,
    'name_original': 'people',
    'name_normalized': 'people',
    'element_id': 'people',
    'description': None,
    'n_columns': 5}],
  'columns': [{'column_index': 1,
    'table_index': 0,
    'table_name': 'perpetrator',
    'name_original': 'Perpetrator_ID',
    'name_normalized': 'perpetrator id',
    'column_type': 'number',
    'element_id': 'perpetrator.Perpetrator_ID',
    'is_primary_key': True,
    'description': None},
   {'column_index': 2,
    'table_index': 0,
    'table_name': 'perpetrator',
    'name_original': 'People_ID',
    'name_normalized': 'people id',
    'column_type': 'number',
    'element_id': 'perpetr

## 3. Validación de particiones e integridad

In [12]:
from src.dataset_validator import ValidationReport, validate_master, validate_partitions

report = ValidationReport()
validate_master(master, catalog, report)
dist = validate_partitions(master, report)
summary_df = report.summary_frame()
display(dist)
summary_df[summary_df.severity != "ok"]

,source,split,n_examples,n_dbs
0,BIRD,test,1432,14
1,BIRD,train,7996,55
2,BIRD,validation,1534,11
3,CATASTRO,test_transfer,80,1
4,SPIDER,train,7000,140
5,SPIDER,validation,1034,20


,category,check,source,count,severity,note
4,integridad,registros_duplicados_pregunta_sql,BIRD,12,warning,"marcados, no eliminados"
11,integridad,evidence_vacia,BIRD,793,info,conservadas por diseño
26,integridad,registros_duplicados_pregunta_sql,SPIDER,18,warning,"marcados, no eliminados"


In [13]:
# Comprobación explícita de disjunción de bases por partición
for source, df in master.groupby("source"):
    splits = {s: set(g.db_id) for s, g in df.groupby("split")}
    names = sorted(splits)
    inter = {f"{a}∩{b}": len(splits[a] & splits[b]) for i, a in enumerate(names) for b in names[i + 1:]}
    print(source, {k: len(v) for k, v in splits.items()}, "intersecciones:", inter)

BIRD {'test': 14, 'train': 55, 'validation': 11} intersecciones: {'test∩train': 0, 'test∩validation': 0, 'train∩validation': 0}
CATASTRO {'test_transfer': 1} intersecciones: {}
SPIDER {'train': 140, 'validation': 20} intersecciones: {'train∩validation': 0}


## 4. Resumen

* `master_dataset.parquet`: 19.076 registros (Spider 8.034, BIRD 10.962, Catastro 80) con 16 columnas; identificadores únicos; 30 duplicados marcados.
* `bird_db_split.json`: 14 bases reservadas (1.432 ejemplos) de forma reproducible.
* `schemas.json`: 247 esquemas (166 Spider, 80 BIRD, 1 Catastro) con tablas, columnas, tipos, PK, FK, relaciones y DDL.
* Las particiones son disjuntas por `db_id` dentro de cada fuente y Catastro está íntegramente en `test_transfer`.
